In [1]:
# ==================================================
# ADIM 1: MUTAG Deneyi İçin Gerekli Kütüphaneler
# ==================================================

import os
import time
import math
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

print("ADIM 1 tamamlandı.")
print("Temel kütüphaneler başarıyla yüklendi.")

ADIM 1 tamamlandı.
Temel kütüphaneler başarıyla yüklendi.


In [2]:
# ==================================================
# ADIM 2: PyTorch Geometric Kontrolü ve MUTAG Yükleme
# ==================================================

try:
    import torch
    from torch_geometric.datasets import TUDataset
    from torch_geometric.utils import to_networkx
    
    print("PyTorch sürümü:", torch.__version__)
    print("PyTorch Geometric başarıyla yüklü.")
    
except ImportError as e:
    print("PyTorch Geometric yüklü değil.")
    print("Hata:", e)

PyTorch sürümü: 2.2.2
PyTorch Geometric başarıyla yüklü.


In [3]:
# ==================================================
# ADIM 3B: MUTAG Dosyalarının Yerini Kontrol Etme
# ==================================================

import os

possible_paths = [
    "data/TUDataset/MUTAG/raw",
    "data/TUDataset/MUTAG",
    "MUTAG",
    "data/MUTAG"
]

for path in possible_paths:
    print("\nKontrol edilen klasör:", os.path.abspath(path))
    
    if os.path.exists(path):
        print("Klasör bulundu.")
        print("İçindeki dosyalar:")
        for file in os.listdir(path):
            print(" -", file)
    else:
        print("Klasör bulunamadı.")


Kontrol edilen klasör: /Users/leyla/Desktop/Github/Tez/attribute_index_thesis/mutag/data/TUDataset/MUTAG/raw
Klasör bulundu.
İçindeki dosyalar:
 - MUTAG_A.txt
 - MUTAG_graph_labels.txt
 - MUTAG_node_labels.txt
 - MUTAG_graph_indicator.txt
 - README.txt
 - MUTAG_edge_labels.txt

Kontrol edilen klasör: /Users/leyla/Desktop/Github/Tez/attribute_index_thesis/mutag/data/TUDataset/MUTAG
Klasör bulundu.
İçindeki dosyalar:
 - .DS_Store
 - processed
 - raw

Kontrol edilen klasör: /Users/leyla/Desktop/Github/Tez/attribute_index_thesis/mutag/MUTAG
Klasör bulunamadı.

Kontrol edilen klasör: /Users/leyla/Desktop/Github/Tez/attribute_index_thesis/mutag/data/MUTAG
Klasör bulunamadı.


In [4]:
# ==================================================
# ADIM 4: MUTAG Dosyalarını Manuel Okuma
# ==================================================

import os
import pandas as pd

RAW_DIR = "data/TUDataset/MUTAG/raw"

# Dosya yolları
path_A = os.path.join(RAW_DIR, "MUTAG_A.txt")
path_graph_indicator = os.path.join(RAW_DIR, "MUTAG_graph_indicator.txt")
path_graph_labels = os.path.join(RAW_DIR, "MUTAG_graph_labels.txt")
path_node_labels = os.path.join(RAW_DIR, "MUTAG_node_labels.txt")
path_edge_labels = os.path.join(RAW_DIR, "MUTAG_edge_labels.txt")

# Kenar listesi: her satır "u, v"
edges_df = pd.read_csv(path_A, header=None, names=["source", "target"])

# Her düğümün hangi grafa ait olduğunu gösterir
graph_indicator = pd.read_csv(path_graph_indicator, header=None, names=["graph_id"])

# Her grafın sınıf etiketi
graph_labels = pd.read_csv(path_graph_labels, header=None, names=["label"])

# Düğüm etiketleri
node_labels = pd.read_csv(path_node_labels, header=None, names=["node_label"])

# Kenar etiketleri
edge_labels = pd.read_csv(path_edge_labels, header=None, names=["edge_label"])

print("ADIM 4 tamamlandı.")
print("Kenar sayısı:", len(edges_df))
print("Düğüm sayısı:", len(graph_indicator))
print("Graf sayısı:", len(graph_labels))
print("Düğüm etiketi sayısı:", len(node_labels))
print("Kenar etiketi sayısı:", len(edge_labels))

print("\nİlk 5 kenar:")
display(edges_df.head())

print("\nİlk 5 graf etiketi:")
display(graph_labels.head())

ADIM 4 tamamlandı.
Kenar sayısı: 7442
Düğüm sayısı: 3371
Graf sayısı: 188
Düğüm etiketi sayısı: 3371
Kenar etiketi sayısı: 7442

İlk 5 kenar:


,source,target
0,2,1
1,1,2
2,3,2
3,2,3
4,4,3



İlk 5 graf etiketi:


,label
0,1
1,-1
2,-1
3,1
4,-1


In [5]:
# ==================================================
# ADIM 5: TXT Dosyalarından NetworkX Graf Listesi Oluşturma
# ==================================================

import networkx as nx
from collections import defaultdict

start_time = time.time()

# Her düğümün hangi grafa ait olduğunu belirleyelim.
# Düğüm id'leri MUTAG dosyasında 1'den başlıyor.
node_to_graph = {}

for node_index, graph_id in enumerate(graph_indicator["graph_id"], start=1):
    node_to_graph[node_index] = int(graph_id)

# Graf id -> düğüm listesi
graph_to_nodes = defaultdict(list)

for node_id, graph_id in node_to_graph.items():
    graph_to_nodes[graph_id].append(node_id)

# Graf id -> kenar listesi
graph_to_edges = defaultdict(list)

for _, row in edges_df.iterrows():
    u = int(row["source"])
    v = int(row["target"])
    
    graph_u = node_to_graph[u]
    graph_v = node_to_graph[v]
    
    # Güvenlik kontrolü: Bir kenarın iki ucu aynı grafa ait olmalı
    if graph_u == graph_v:
        graph_to_edges[graph_u].append((u, v))
    else:
        print("UYARI: Farklı graflar arasında kenar bulundu:", u, v)

# Her graf için NetworkX nesnesi oluşturalım
mutag_graphs = []

for graph_id in range(1, len(graph_labels) + 1):
    G = nx.Graph()
    
    # Düğümleri ekle
    G.add_nodes_from(graph_to_nodes[graph_id])
    
    # Kenarları ekle
    G.add_edges_from(graph_to_edges[graph_id])
    
    # Etiketler MUTAG'da genelde -1 ve 1 şeklindedir.
    # Makine öğrenmesi için 0 ve 1'e çevireceğiz.
    original_label = int(graph_labels.iloc[graph_id - 1]["label"])
    label = 1 if original_label == 1 else 0
    
    mutag_graphs.append({
        "graph_id": graph_id,
        "graph": G,
        "label": label,
        "original_label": original_label
    })

elapsed = time.time() - start_time

print("ADIM 5 tamamlandı.")
print("Oluşturulan graf sayısı:", len(mutag_graphs))
print("İlk graf düğüm sayısı:", mutag_graphs[0]["graph"].number_of_nodes())
print("İlk graf kenar sayısı:", mutag_graphs[0]["graph"].number_of_edges())
print("İlk graf etiketi:", mutag_graphs[0]["label"])
print(f"Çalışma süresi: {elapsed:.2f} saniye")

ADIM 5 tamamlandı.
Oluşturulan graf sayısı: 188
İlk graf düğüm sayısı: 17
İlk graf kenar sayısı: 19
İlk graf etiketi: 1
Çalışma süresi: 0.20 saniye


In [6]:
# ==================================================
# ADIM 6: MUTAG Graf İstatistiklerini Kontrol Etme
# ==================================================

stats_rows = []

for item in mutag_graphs:
    G = item["graph"]
    
    stats_rows.append({
        "graph_id": item["graph_id"],
        "num_nodes": G.number_of_nodes(),
        "num_edges": G.number_of_edges(),
        "label": item["label"],
        "original_label": item["original_label"],
        "is_connected": nx.is_connected(G) if G.number_of_nodes() > 0 else False
    })

stats_df = pd.DataFrame(stats_rows)

print("ADIM 6 tamamlandı.")
print("Toplam graf sayısı:", len(stats_df))
print("\nSınıf dağılımı:")
print(stats_df["label"].value_counts().sort_index())

print("\nTemel istatistikler:")
display(stats_df[["num_nodes", "num_edges"]].describe())

print("\nBağlantılı graf sayısı:")
print(stats_df["is_connected"].value_counts())

print("\nİlk 10 graf:")
display(stats_df.head(10))

ADIM 6 tamamlandı.
Toplam graf sayısı: 188

Sınıf dağılımı:
label
0     63
1    125
Name: count, dtype: int64

Temel istatistikler:


,num_nodes,num_edges
count,188.000000,188.000000
mean,17.930851,19.792553
std,4.587883,5.699662
min,10.000000,10.000000
25%,14.000000,14.000000
50%,17.500000,19.000000
75%,22.000000,25.000000
max,28.000000,33.000000



Bağlantılı graf sayısı:
is_connected
True    188
Name: count, dtype: int64

İlk 10 graf:


,graph_id,num_nodes,num_edges,label,original_label,is_connected
0,1,17,19,1,1,True
1,2,13,14,0,-1,True
2,3,13,14,0,-1,True
3,4,19,22,1,1,True
4,5,11,11,0,-1,True
5,6,28,31,1,1,True
6,7,16,17,0,-1,True
7,8,20,22,1,1,True
8,9,12,13,0,-1,True
9,10,17,19,1,1,True


In [7]:
# ==================================================
# ADIM 7: Global Graf İndeks Fonksiyonlarını Tanımlama
# ==================================================

def wiener_index(G):
    """
    Wiener indeksi:
    Graf içindeki tüm düğüm çiftleri arasındaki en kısa yol mesafelerinin toplamı.
    """
    lengths = dict(nx.all_pairs_shortest_path_length(G))
    nodes = list(G.nodes())
    
    total = 0
    
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            total += lengths[nodes[i]][nodes[j]]
    
    return total


def zagreb_indices(G):
    """
    Birinci ve ikinci Zagreb indeksleri.
    M1 = sum d(v)^2
    M2 = sum d(u)*d(v), uv kenarları üzerinde
    """
    degrees = dict(G.degree())
    
    M1 = sum(d ** 2 for d in degrees.values())
    M2 = sum(degrees[u] * degrees[v] for u, v in G.edges())
    
    return M1, M2


def randic_index(G):
    """
    Randić indeksi:
    R = sum_{uv in E} 1 / sqrt(d(u)*d(v))
    """
    degrees = dict(G.degree())
    
    total = 0
    
    for u, v in G.edges():
        du = degrees[u]
        dv = degrees[v]
        
        if du > 0 and dv > 0:
            total += 1 / math.sqrt(du * dv)
    
    return total


def sombor_index(G):
    """
    Sombor indeksi:
    SO = sum_{uv in E} sqrt(d(u)^2 + d(v)^2)
    """
    degrees = dict(G.degree())
    
    total = 0
    
    for u, v in G.edges():
        total += math.sqrt(degrees[u] ** 2 + degrees[v] ** 2)
    
    return total


def estrada_index(G):
    """
    Estrada indeksi:
    EE = sum exp(lambda_i)
    lambda_i: komşuluk matrisinin özdeğerleri
    """
    A = nx.to_numpy_array(G)
    eigenvalues = np.linalg.eigvals(A)
    
    return float(np.sum(np.exp(eigenvalues)).real)


def gutman_index(G):
    """
    Gutman indeksi:
    Gut = sum_{u<v} d(u)*d(v)*dist(u,v)
    """
    degrees = dict(G.degree())
    lengths = dict(nx.all_pairs_shortest_path_length(G))
    nodes = list(G.nodes())
    
    total = 0
    
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            u = nodes[i]
            v = nodes[j]
            total += degrees[u] * degrees[v] * lengths[u][v]
    
    return total


print("ADIM 7 tamamlandı.")
print("Wiener, Zagreb, Randić, Sombor, Estrada ve Gutman fonksiyonları tanımlandı.")

ADIM 7 tamamlandı.
Wiener, Zagreb, Randić, Sombor, Estrada ve Gutman fonksiyonları tanımlandı.


In [8]:
# ==================================================
# ADIM 8: İndeks Fonksiyonlarını İlk Graf Üzerinde Test Etme
# ==================================================

G0 = mutag_graphs[0]["graph"]

W = wiener_index(G0)
M1, M2 = zagreb_indices(G0)
R = randic_index(G0)
SO = sombor_index(G0)
EE = estrada_index(G0)
Gut = gutman_index(G0)

print("ADIM 8 tamamlandı.")
print("İlk graf için hesaplanan indeksler:")
print("Düğüm sayısı:", G0.number_of_nodes())
print("Kenar sayısı:", G0.number_of_edges())
print("Wiener indeksi:", W)
print("Zagreb M1:", M1)
print("Zagreb M2:", M2)
print("Randić indeksi:", R)
print("Sombor indeksi:", SO)
print("Estrada indeksi:", EE)
print("Gutman indeksi:", Gut)

ADIM 8 tamamlandı.
İlk graf için hesaplanan indeksler:
Düğüm sayısı: 17
Kenar sayısı: 19
Wiener indeksi: 492
Zagreb M1: 92
Zagreb M2: 110
Randić indeksi: 8.25402019542349
Sombor indeksi: 66.28166389625676
Estrada indeksi: 43.13990901421484
Gutman indeksi: 2253


In [9]:
# ==================================================
# ADIM 9: Tüm MUTAG Grafları İçin Global İndeksleri Hesaplama
# ==================================================

start_time = time.time()

rows = []

for item in mutag_graphs:
    graph_id = item["graph_id"]
    G = item["graph"]
    label = item["label"]
    original_label = item["original_label"]
    
    # Temel graf özellikleri
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    density = nx.density(G)
    avg_degree = sum(dict(G.degree()).values()) / num_nodes
    
    # Global graf indeksleri
    W = wiener_index(G)
    M1, M2 = zagreb_indices(G)
    R = randic_index(G)
    SO = sombor_index(G)
    EE = estrada_index(G)
    Gut = gutman_index(G)
    
    rows.append({
        "graph_id": graph_id,
        "num_nodes": num_nodes,
        "num_edges": num_edges,
        "density": density,
        "avg_degree": avg_degree,
        "wiener": W,
        "zagreb_m1": M1,
        "zagreb_m2": M2,
        "randic": R,
        "sombor": SO,
        "estrada": EE,
        "gutman": Gut,
        "label": label,
        "original_label": original_label
    })

mutag_df = pd.DataFrame(rows)

elapsed = time.time() - start_time

print("ADIM 9 tamamlandı.")
print("Oluşturulan tablo boyutu:", mutag_df.shape)
print(f"Çalışma süresi: {elapsed:.2f} saniye")

display(mutag_df.head())

ADIM 9 tamamlandı.
Oluşturulan tablo boyutu: (188, 14)
Çalışma süresi: 0.17 saniye


,graph_id,num_nodes,num_edges,density,avg_degree,wiener,zagreb_m1,zagreb_m2,randic,sombor,estrada,gutman,label,original_label
0,1,17,19,0.139706,2.235294,492,92,110,8.254020,66.281664,43.139909,2253,1,1
1,2,13,14,0.179487,2.153846,226,66,77,6.287694,47.616818,32.061284,944,0,-1
2,3,13,14,0.179487,2.153846,226,66,77,6.287694,47.616818,32.061284,944,0,-1
3,4,19,22,0.128655,2.315789,586,110,138,9.270857,78.869551,49.786763,2843,1,1
4,5,11,11,0.200000,2.000000,148,52,58,5.109061,38.385024,25.968481,505,0,-1


In [10]:
# ==================================================
# ADIM 10: MUTAG İndeks Tablosunu Kontrol Etme
# ==================================================

print("ADIM 10 başladı.")

print("\nTablo boyutu:")
print(mutag_df.shape)

print("\nSütunlar:")
print(mutag_df.columns.tolist())

print("\nEksik değer kontrolü:")
print(mutag_df.isnull().sum())

print("\nSayısal özniteliklerin temel istatistikleri:")
display(mutag_df.describe())

print("\nSınıfa göre ortalama indeks değerleri:")
class_means = mutag_df.groupby("label")[
    [
        "num_nodes",
        "num_edges",
        "density",
        "avg_degree",
        "wiener",
        "zagreb_m1",
        "zagreb_m2",
        "randic",
        "sombor",
        "estrada",
        "gutman"
    ]
].mean()

display(class_means)

print("ADIM 10 tamamlandı.")

ADIM 10 başladı.

Tablo boyutu:
(188, 14)

Sütunlar:
['graph_id', 'num_nodes', 'num_edges', 'density', 'avg_degree', 'wiener', 'zagreb_m1', 'zagreb_m2', 'randic', 'sombor', 'estrada', 'gutman', 'label', 'original_label']

Eksik değer kontrolü:
graph_id          0
num_nodes         0
num_edges         0
density           0
avg_degree        0
wiener            0
zagreb_m1         0
zagreb_m2         0
randic            0
sombor            0
estrada           0
gutman            0
label             0
original_label    0
dtype: int64

Sayısal özniteliklerin temel istatistikleri:


,graph_id,num_nodes,num_edges,density,avg_degree,wiener,zagreb_m1,zagreb_m2,randic,sombor,estrada,gutman,label,original_label
count,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000,188.000000
mean,94.500000,17.930851,19.792553,0.138454,2.188772,628.095745,97.329787,117.462766,8.593993,70.494745,45.359727,2803.026596,0.664894,0.329787
std,54.415071,4.587883,5.699662,0.035146,0.109645,386.363329,30.444089,40.758631,2.234196,21.780116,12.728944,1841.048958,0.473288,0.946576
min,1.000000,10.000000,10.000000,0.082011,2.000000,114.000000,46.000000,50.000000,4.698377,33.668498,23.464824,396.000000,0.000000,-1.000000
25%,47.750000,14.000000,14.000000,0.108225,2.125000,279.500000,68.000000,80.000000,6.525919,49.644952,33.040463,1004.000000,0.000000,-1.000000
50%,94.500000,17.500000,19.000000,0.131536,2.200000,585.000000,94.000000,113.000000,8.423122,68.169763,43.675974,2452.000000,1.000000,1.000000
75%,141.250000,22.000000,25.000000,0.164835,2.272727,910.500000,122.000000,145.250000,10.545923,89.497971,56.947531,4084.000000,1.000000,1.000000
max,188.000000,28.000000,33.000000,0.222222,2.444444,1694.000000,172.000000,224.000000,13.558551,123.130276,74.600990,8291.000000,1.000000,1.000000



Sınıfa göre ortalama indeks değerleri:


,num_nodes,num_edges,density,avg_degree,wiener,zagreb_m1,zagreb_m2,randic,sombor,estrada,gutman
label,,,,,,,,,,,
0,13.936508,14.619048,0.169006,2.089930,331.333333,69.587302,80.253968,6.616339,50.724318,33.917065,1316.428571
1,19.944000,22.400000,0.123056,2.238588,777.664000,111.312000,136.216000,9.590731,80.459041,51.126828,3552.272000


ADIM 10 tamamlandı.


In [11]:
# ==================================================
# ADIM 11: MUTAG İçin Öznitelik Setlerini Oluşturma
# ==================================================

# 1) Sadece temel graf özellikleri
features_basic = [
    "num_nodes",
    "num_edges",
    "density",
    "avg_degree"
]

# 2) Sadece global graf indeksleri
features_indices = [
    "wiener",
    "zagreb_m1",
    "zagreb_m2",
    "randic",
    "sombor",
    "estrada",
    "gutman"
]

# 3) Temel özellikler + global graf indeksleri
features_all = features_basic + features_indices

feature_sets = {
    "Basic": features_basic,
    "Graph_Indices": features_indices,
    "Basic_Plus_Graph_Indices": features_all
}

target_col = "label"

print("ADIM 11 tamamlandı.")
print("Tanımlanan öznitelik setleri:\n")

for set_name, cols in feature_sets.items():
    print(f"{set_name}: {len(cols)} öznitelik")
    print(cols)
    print("-" * 60)

print("Hedef değişken:", target_col)
print("Sınıf dağılımı:")
print(mutag_df[target_col].value_counts().sort_index())

ADIM 11 tamamlandı.
Tanımlanan öznitelik setleri:

Basic: 4 öznitelik
['num_nodes', 'num_edges', 'density', 'avg_degree']
------------------------------------------------------------
Graph_Indices: 7 öznitelik
['wiener', 'zagreb_m1', 'zagreb_m2', 'randic', 'sombor', 'estrada', 'gutman']
------------------------------------------------------------
Basic_Plus_Graph_Indices: 11 öznitelik
['num_nodes', 'num_edges', 'density', 'avg_degree', 'wiener', 'zagreb_m1', 'zagreb_m2', 'randic', 'sombor', 'estrada', 'gutman']
------------------------------------------------------------
Hedef değişken: label
Sınıf dağılımı:
label
0     63
1    125
Name: count, dtype: int64


In [12]:
# ==================================================
# ADIM 12A: XGBoost Kontrolü
# ==================================================

try:
    from xgboost import XGBClassifier
    print("XGBoost başarıyla yüklü.")
except ImportError as e:
    print("XGBoost yüklü değil.")
    print("Hata:", e)

XGBoost başarıyla yüklü.


In [13]:
# ==================================================
# ADIM 12B: Train-Test Ayrımı ile RF, XGBoost, SVM ve MLP Denemesi
# ==================================================

start_time = time.time()

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ),
    
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        reg_alpha=0.1,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1
    ),
    
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    
    "MLP": MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=RANDOM_STATE
    )
}

train_test_results = []

for set_name, feature_cols in feature_sets.items():
    print("=" * 75)
    print(f"Öznitelik seti: {set_name} | Öznitelik sayısı: {len(feature_cols)}")
    print("=" * 75)
    
    X = mutag_df[feature_cols]
    y = mutag_df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y
    )
    
    for model_name, model in models.items():
        print(f"Model eğitiliyor: {model_name} ...")
        
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
        
        model_start = time.time()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        model_time = time.time() - model_start
        
        acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average="macro")
        
        train_test_results.append({
            "Feature_Set": set_name,
            "Model": model_name,
            "Accuracy": acc,
            "Macro_F1": macro_f1,
            "Time_sec": model_time
        })
        
        print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Süre: {model_time:.2f} sn")
    
    print()

train_test_results_df = pd.DataFrame(train_test_results)

elapsed = time.time() - start_time

print("ADIM 12B tamamlandı.")
print(f"Toplam çalışma süresi: {elapsed:.2f} saniye")

display(train_test_results_df)

Öznitelik seti: Basic | Öznitelik sayısı: 4
Model eğitiliyor: RandomForest ...
Accuracy: 0.8085 | Macro-F1: 0.7899 | Süre: 0.28 sn
Model eğitiliyor: XGBoost ...
Accuracy: 0.8085 | Macro-F1: 0.7899 | Süre: 0.02 sn
Model eğitiliyor: SVM ...
Accuracy: 0.8298 | Macro-F1: 0.8157 | Süre: 0.01 sn
Model eğitiliyor: MLP ...
Accuracy: 0.8085 | Macro-F1: 0.7899 | Süre: 0.19 sn

Öznitelik seti: Graph_Indices | Öznitelik sayısı: 7
Model eğitiliyor: RandomForest ...
Accuracy: 0.7660 | Macro-F1: 0.7432 | Süre: 0.23 sn
Model eğitiliyor: XGBoost ...
Accuracy: 0.8085 | Macro-F1: 0.7899 | Süre: 0.02 sn
Model eğitiliyor: SVM ...
Accuracy: 0.8085 | Macro-F1: 0.7899 | Süre: 0.00 sn
Model eğitiliyor: MLP ...
Accuracy: 0.7872 | Macro-F1: 0.7696 | Süre: 0.35 sn

Öznitelik seti: Basic_Plus_Graph_Indices | Öznitelik sayısı: 11
Model eğitiliyor: RandomForest ...
Accuracy: 0.7660 | Macro-F1: 0.7432 | Süre: 0.24 sn
Model eğitiliyor: XGBoost ...
Accuracy: 0.8085 | Macro-F1: 0.7899 | Süre: 0.02 sn
Model eğitiliyor: S

,Feature_Set,Model,Accuracy,Macro_F1,Time_sec
0,Basic,RandomForest,0.808511,0.789866,0.280687
1,Basic,XGBoost,0.808511,0.789866,0.023696
2,Basic,SVM,0.829787,0.815686,0.006893
3,Basic,MLP,0.808511,0.789866,0.193818
4,Graph_Indices,RandomForest,0.765957,0.743169,0.234250
5,Graph_Indices,XGBoost,0.808511,0.789866,0.016634
6,Graph_Indices,SVM,0.808511,0.789866,0.003782
7,Graph_Indices,MLP,0.787234,0.769608,0.354879
8,Basic_Plus_Graph_Indices,RandomForest,0.765957,0.743169,0.238773
9,Basic_Plus_Graph_Indices,XGBoost,0.808511,0.789866,0.017798


In [14]:
# ==================================================
# ADIM 13: 5-Fold Cross Validation ile Model Değerlendirme
# ==================================================

start_time = time.time()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro"
}

cv_results = []

for set_name, feature_cols in feature_sets.items():
    print("=" * 75)
    print(f"5-Fold CV | Öznitelik seti: {set_name} | Öznitelik sayısı: {len(feature_cols)}")
    print("=" * 75)
    
    X = mutag_df[feature_cols]
    y = mutag_df[target_col]
    
    for model_name, model in models.items():
        print(f"Model değerlendiriliyor: {model_name} ...")
        
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
        
        scores = cross_validate(
            pipe,
            X,
            y,
            cv=cv,
            scoring=scoring,
            return_train_score=False,
            n_jobs=1
        )
        
        acc_mean = scores["test_accuracy"].mean()
        acc_std = scores["test_accuracy"].std()
        f1_mean = scores["test_macro_f1"].mean()
        f1_std = scores["test_macro_f1"].std()
        fit_time_mean = scores["fit_time"].mean()
        
        cv_results.append({
            "Feature_Set": set_name,
            "Model": model_name,
            "Accuracy_Mean": acc_mean,
            "Accuracy_Std": acc_std,
            "Macro_F1_Mean": f1_mean,
            "Macro_F1_Std": f1_std,
            "Fit_Time_Mean": fit_time_mean
        })
        
        print(
            f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f} | "
            f"Macro-F1: {f1_mean:.4f} ± {f1_std:.4f} | "
            f"Ortalama süre: {fit_time_mean:.2f} sn"
        )
    
    print()

cv_results_df = pd.DataFrame(cv_results)

elapsed = time.time() - start_time

print("ADIM 13 tamamlandı.")
print(f"Toplam çalışma süresi: {elapsed:.2f} saniye")

display(cv_results_df)

5-Fold CV | Öznitelik seti: Basic | Öznitelik sayısı: 4
Model değerlendiriliyor: RandomForest ...
Accuracy: 0.8459 ± 0.0422 | Macro-F1: 0.8254 ± 0.0538 | Ortalama süre: 0.22 sn
Model değerlendiriliyor: XGBoost ...
Accuracy: 0.8670 ± 0.0473 | Macro-F1: 0.8480 ± 0.0596 | Ortalama süre: 0.01 sn
Model değerlendiriliyor: SVM ...
Accuracy: 0.8671 ± 0.0373 | Macro-F1: 0.8546 ± 0.0416 | Ortalama süre: 0.00 sn
Model değerlendiriliyor: MLP ...
Accuracy: 0.8724 ± 0.0512 | Macro-F1: 0.8542 ± 0.0642 | Ortalama süre: 0.18 sn

5-Fold CV | Öznitelik seti: Graph_Indices | Öznitelik sayısı: 7
Model değerlendiriliyor: RandomForest ...
Accuracy: 0.8508 ± 0.0370 | Macro-F1: 0.8272 ± 0.0465 | Ortalama süre: 0.24 sn
Model değerlendiriliyor: XGBoost ...
Accuracy: 0.8509 ± 0.0465 | Macro-F1: 0.8225 ± 0.0711 | Ortalama süre: 0.01 sn
Model değerlendiriliyor: SVM ...
Accuracy: 0.8829 ± 0.0433 | Macro-F1: 0.8691 ± 0.0499 | Ortalama süre: 0.00 sn
Model değerlendiriliyor: MLP ...
Accuracy: 0.8777 ± 0.0274 | Macro-F1

,Feature_Set,Model,Accuracy_Mean,Accuracy_Std,Macro_F1_Mean,Macro_F1_Std,Fit_Time_Mean
0,Basic,RandomForest,0.845946,0.042151,0.825374,0.053834,0.223832
1,Basic,XGBoost,0.866999,0.047268,0.847983,0.059597,0.012925
2,Basic,SVM,0.867141,0.037255,0.854641,0.041607,0.003029
3,Basic,MLP,0.872404,0.051189,0.854207,0.064209,0.176573
4,Graph_Indices,RandomForest,0.850782,0.037024,0.827238,0.046451,0.239621
5,Graph_Indices,XGBoost,0.850925,0.046484,0.822489,0.071128,0.014859
6,Graph_Indices,SVM,0.882930,0.043262,0.869095,0.049931,0.003346
7,Graph_Indices,MLP,0.877667,0.027388,0.861854,0.035417,0.230402
8,Basic_Plus_Graph_Indices,RandomForest,0.845519,0.027237,0.821853,0.036620,0.229155
9,Basic_Plus_Graph_Indices,XGBoost,0.856188,0.036711,0.834806,0.047343,0.015451


In [15]:
# ==================================================
# ADIM 14: CV Sonuçlarını Sıralama ve En İyi Modeli Seçme
# ==================================================

cv_results_sorted = cv_results_df.sort_values(
    by=["Accuracy_Mean", "Macro_F1_Mean"],
    ascending=False
).reset_index(drop=True)

print("ADIM 14 tamamlandı.")
print("5-Fold Cross Validation sonuçları başarıya göre sıralandı.\n")

display(cv_results_sorted)

best_result = cv_results_sorted.iloc[0]

print("\nEn iyi sonuç:")
print("Öznitelik seti:", best_result["Feature_Set"])
print("Model:", best_result["Model"])
print(f"Accuracy: {best_result['Accuracy_Mean']:.4f} ± {best_result['Accuracy_Std']:.4f}")
print(f"Macro-F1: {best_result['Macro_F1_Mean']:.4f} ± {best_result['Macro_F1_Std']:.4f}")

ADIM 14 tamamlandı.
5-Fold Cross Validation sonuçları başarıya göre sıralandı.



,Feature_Set,Model,Accuracy_Mean,Accuracy_Std,Macro_F1_Mean,Macro_F1_Std,Fit_Time_Mean
0,Graph_Indices,SVM,0.882930,0.043262,0.869095,0.049931,0.003346
1,Graph_Indices,MLP,0.877667,0.027388,0.861854,0.035417,0.230402
2,Basic_Plus_Graph_Indices,SVM,0.872404,0.030955,0.859744,0.036429,0.003077
3,Basic,MLP,0.872404,0.051189,0.854207,0.064209,0.176573
4,Basic,SVM,0.867141,0.037255,0.854641,0.041607,0.003029
5,Basic,XGBoost,0.866999,0.047268,0.847983,0.059597,0.012925
6,Basic_Plus_Graph_Indices,XGBoost,0.856188,0.036711,0.834806,0.047343,0.015451
7,Graph_Indices,XGBoost,0.850925,0.046484,0.822489,0.071128,0.014859
8,Graph_Indices,RandomForest,0.850782,0.037024,0.827238,0.046451,0.239621
9,Basic_Plus_Graph_Indices,MLP,0.845946,0.024963,0.826907,0.036116,0.319090



En iyi sonuç:
Öznitelik seti: Graph_Indices
Model: SVM
Accuracy: 0.8829 ± 0.0433
Macro-F1: 0.8691 ± 0.0499


In [16]:
# ==================================================
# ADIM 15: Random Forest ve XGBoost Feature Importance
# ==================================================

start_time = time.time()

# En iyi sonuç veren ve tez açısından en anlamlı öznitelik seti
selected_feature_set_name = "Graph_Indices"
selected_features = feature_sets[selected_feature_set_name]

X = mutag_df[selected_features]
y = mutag_df[target_col]

# Train-test ayrımı
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

# Ölçekleme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest modeli
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)

# XGBoost modeli
xgb_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.1,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=1
)

# Modelleri eğit
rf_model.fit(X_train_scaled, y_train)
xgb_model.fit(X_train_scaled, y_train)

# Tahminler
rf_pred = rf_model.predict(X_test_scaled)
xgb_pred = xgb_model.predict(X_test_scaled)

print("ADIM 15 başladı.")
print("Kullanılan öznitelik seti:", selected_feature_set_name)
print("Öznitelikler:", selected_features)

print("\nRandom Forest test sonucu:")
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("Macro-F1:", round(f1_score(y_test, rf_pred, average="macro"), 4))

print("\nXGBoost test sonucu:")
print("Accuracy:", round(accuracy_score(y_test, xgb_pred), 4))
print("Macro-F1:", round(f1_score(y_test, xgb_pred, average="macro"), 4))

# Feature importance tabloları
rf_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

xgb_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\nRandom Forest Feature Importance:")
display(rf_importance_df)

print("\nXGBoost Feature Importance:")
display(xgb_importance_df)

elapsed = time.time() - start_time

print("ADIM 15 tamamlandı.")
print(f"Çalışma süresi: {elapsed:.2f} saniye")

ADIM 15 başladı.
Kullanılan öznitelik seti: Graph_Indices
Öznitelikler: ['wiener', 'zagreb_m1', 'zagreb_m2', 'randic', 'sombor', 'estrada', 'gutman']

Random Forest test sonucu:
Accuracy: 0.766
Macro-F1: 0.7432

XGBoost test sonucu:
Accuracy: 0.8085
Macro-F1: 0.7899

Random Forest Feature Importance:


,Feature,Importance
0,sombor,0.228127
1,zagreb_m1,0.182991
2,gutman,0.180915
3,zagreb_m2,0.166131
4,estrada,0.097124
5,wiener,0.079280
6,randic,0.065430



XGBoost Feature Importance:


,Feature,Importance
0,zagreb_m1,0.388114
1,gutman,0.157718
2,zagreb_m2,0.136363
3,sombor,0.132681
4,randic,0.101091
5,wiener,0.043506
6,estrada,0.040529


ADIM 15 tamamlandı.
Çalışma süresi: 0.48 saniye


In [17]:
# ==================================================
# ADIM 16: SVM İçin Permutation Importance
# ==================================================

from sklearn.inspection import permutation_importance

start_time = time.time()

# En iyi sonuç veren öznitelik seti
selected_feature_set_name = "Graph_Indices"
selected_features = feature_sets[selected_feature_set_name]

X = mutag_df[selected_features]
y = mutag_df[target_col]

# Train-test ayrımı
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

# SVM Pipeline
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

# Modeli eğit
svm_pipe.fit(X_train, y_train)

# Test başarısı
y_pred = svm_pipe.predict(X_test)

svm_acc = accuracy_score(y_test, y_pred)
svm_macro_f1 = f1_score(y_test, y_pred, average="macro")

print("ADIM 16 başladı.")
print("Kullanılan model: SVM")
print("Kullanılan öznitelik seti:", selected_feature_set_name)

print("\nSVM test sonucu:")
print("Accuracy:", round(svm_acc, 4))
print("Macro-F1:", round(svm_macro_f1, 4))

# Permutation importance
perm_result = permutation_importance(
    svm_pipe,
    X_test,
    y_test,
    scoring="accuracy",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=1
)

svm_perm_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_Std": perm_result.importances_std
}).sort_values(by="Importance_Mean", ascending=False).reset_index(drop=True)

print("\nSVM Permutation Importance:")
display(svm_perm_importance_df)

elapsed = time.time() - start_time

print("ADIM 16 tamamlandı.")
print(f"Çalışma süresi: {elapsed:.2f} saniye")

ADIM 16 başladı.
Kullanılan model: SVM
Kullanılan öznitelik seti: Graph_Indices

SVM test sonucu:
Accuracy: 0.8085
Macro-F1: 0.7899

SVM Permutation Importance:


,Feature,Importance_Mean,Importance_Std
0,zagreb_m2,0.038298,0.044086
1,sombor,0.014184,0.030751
2,zagreb_m1,0.014184,0.034889
3,gutman,0.005674,0.021929
4,estrada,0.004255,0.023565
5,wiener,-0.011348,0.024404
6,randic,-0.013475,0.017787


ADIM 16 tamamlandı.
Çalışma süresi: 0.35 saniye


In [18]:
# ==================================================
# ADIM 17: 5-Fold Cross-Validation Tabanlı Classification Report
# ==================================================

from sklearn.model_selection import cross_val_predict

start_time = time.time()

best_feature_set_name = "Graph_Indices"
best_features = feature_sets[best_feature_set_name]

X = mutag_df[best_features]
y = mutag_df[target_col]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

# Her örnek, kendisinin içinde olmadığı fold'da tahmin edilir
y_pred_cv = cross_val_predict(
    best_model,
    X,
    y,
    cv=cv,
    n_jobs=1
)

acc = accuracy_score(y, y_pred_cv)
macro_f1 = f1_score(y, y_pred_cv, average="macro")

print("ADIM 17 tamamlandı.")
print("Değerlendirme yöntemi: 5-Fold Cross-Validation Tahminleri")
print("Model: SVM")
print("Öznitelik seti:", best_feature_set_name)
print("Accuracy:", round(acc, 4))
print("Macro-F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(classification_report(y, y_pred_cv, target_names=["Sınıf 0", "Sınıf 1"]))

cm = confusion_matrix(y, y_pred_cv)

cm_df = pd.DataFrame(
    cm,
    index=["Gerçek 0", "Gerçek 1"],
    columns=["Tahmin 0", "Tahmin 1"]
)

print("\nConfusion Matrix:")
display(cm_df)

elapsed = time.time() - start_time
print(f"Çalışma süresi: {elapsed:.2f} saniye")

ADIM 17 tamamlandı.
Değerlendirme yöntemi: 5-Fold Cross-Validation Tahminleri
Model: SVM
Öznitelik seti: Graph_Indices
Accuracy: 0.883
Macro-F1: 0.8707

Classification Report:
              precision    recall  f1-score   support

     Sınıf 0       0.81      0.86      0.83        63
     Sınıf 1       0.93      0.90      0.91       125

    accuracy                           0.88       188
   macro avg       0.87      0.88      0.87       188
weighted avg       0.89      0.88      0.88       188


Confusion Matrix:


,Tahmin 0,Tahmin 1
Gerçek 0,54,9
Gerçek 1,13,112


Çalışma süresi: 0.04 saniye


In [19]:
# ==================================================
# ADIM 18A: 5-Fold CV Sonuçlarını LaTeX Formatına Çevirme
# ==================================================

cv_latex_df = cv_results_df.copy()

# Sayıları tez formatına uygun hale getirelim
cv_latex_df["Accuracy"] = cv_latex_df.apply(
    lambda row: f"{row['Accuracy_Mean']:.4f} $\\pm$ {row['Accuracy_Std']:.4f}",
    axis=1
)

cv_latex_df["Macro-F1"] = cv_latex_df.apply(
    lambda row: f"{row['Macro_F1_Mean']:.4f} $\\pm$ {row['Macro_F1_Std']:.4f}",
    axis=1
)

cv_latex_df = cv_latex_df[
    [
        "Feature_Set",
        "Model",
        "Accuracy",
        "Macro-F1"
    ]
]

cv_latex_df = cv_latex_df.rename(columns={
    "Feature_Set": "Öznitelik Seti",
    "Model": "Model"
})

print("5-Fold CV Sonuç Tablosu:")
display(cv_latex_df)

print("\nLaTeX kodu:\n")
print(cv_latex_df.to_latex(index=False, escape=False))

5-Fold CV Sonuç Tablosu:


,Öznitelik Seti,Model,Accuracy,Macro-F1
0,Basic,RandomForest,0.8459 $\pm$ 0.0422,0.8254 $\pm$ 0.0538
1,Basic,XGBoost,0.8670 $\pm$ 0.0473,0.8480 $\pm$ 0.0596
2,Basic,SVM,0.8671 $\pm$ 0.0373,0.8546 $\pm$ 0.0416
3,Basic,MLP,0.8724 $\pm$ 0.0512,0.8542 $\pm$ 0.0642
4,Graph_Indices,RandomForest,0.8508 $\pm$ 0.0370,0.8272 $\pm$ 0.0465
5,Graph_Indices,XGBoost,0.8509 $\pm$ 0.0465,0.8225 $\pm$ 0.0711
6,Graph_Indices,SVM,0.8829 $\pm$ 0.0433,0.8691 $\pm$ 0.0499
7,Graph_Indices,MLP,0.8777 $\pm$ 0.0274,0.8619 $\pm$ 0.0354
8,Basic_Plus_Graph_Indices,RandomForest,0.8455 $\pm$ 0.0272,0.8219 $\pm$ 0.0366
9,Basic_Plus_Graph_Indices,XGBoost,0.8562 $\pm$ 0.0367,0.8348 $\pm$ 0.0473



LaTeX kodu:

\begin{tabular}{llll}
\toprule
Öznitelik Seti & Model & Accuracy & Macro-F1 \\
\midrule
Basic & RandomForest & 0.8459 $\pm$ 0.0422 & 0.8254 $\pm$ 0.0538 \\
Basic & XGBoost & 0.8670 $\pm$ 0.0473 & 0.8480 $\pm$ 0.0596 \\
Basic & SVM & 0.8671 $\pm$ 0.0373 & 0.8546 $\pm$ 0.0416 \\
Basic & MLP & 0.8724 $\pm$ 0.0512 & 0.8542 $\pm$ 0.0642 \\
Graph_Indices & RandomForest & 0.8508 $\pm$ 0.0370 & 0.8272 $\pm$ 0.0465 \\
Graph_Indices & XGBoost & 0.8509 $\pm$ 0.0465 & 0.8225 $\pm$ 0.0711 \\
Graph_Indices & SVM & 0.8829 $\pm$ 0.0433 & 0.8691 $\pm$ 0.0499 \\
Graph_Indices & MLP & 0.8777 $\pm$ 0.0274 & 0.8619 $\pm$ 0.0354 \\
Basic_Plus_Graph_Indices & RandomForest & 0.8455 $\pm$ 0.0272 & 0.8219 $\pm$ 0.0366 \\
Basic_Plus_Graph_Indices & XGBoost & 0.8562 $\pm$ 0.0367 & 0.8348 $\pm$ 0.0473 \\
Basic_Plus_Graph_Indices & SVM & 0.8724 $\pm$ 0.0310 & 0.8597 $\pm$ 0.0364 \\
Basic_Plus_Graph_Indices & MLP & 0.8459 $\pm$ 0.0250 & 0.8269 $\pm$ 0.0361 \\
\bottomrule
\end{tabular}



In [20]:
# ==================================================
# ADIM 18B: Random Forest Feature Importance Tablosu
# ==================================================

rf_latex_df = rf_importance_df.copy()
rf_latex_df["Importance"] = rf_latex_df["Importance"].map(lambda x: f"{x:.4f}")

rf_latex_df = rf_latex_df.rename(columns={
    "Feature": "Öznitelik",
    "Importance": "Önem Değeri"
})

print("Random Forest Feature Importance Tablosu:")
display(rf_latex_df)

print("\nLaTeX kodu:\n")
print(rf_latex_df.to_latex(index=False, escape=False))

Random Forest Feature Importance Tablosu:


,Öznitelik,Önem Değeri
0,sombor,0.2281
1,zagreb_m1,0.1830
2,gutman,0.1809
3,zagreb_m2,0.1661
4,estrada,0.0971
5,wiener,0.0793
6,randic,0.0654



LaTeX kodu:

\begin{tabular}{ll}
\toprule
Öznitelik & Önem Değeri \\
\midrule
sombor & 0.2281 \\
zagreb_m1 & 0.1830 \\
gutman & 0.1809 \\
zagreb_m2 & 0.1661 \\
estrada & 0.0971 \\
wiener & 0.0793 \\
randic & 0.0654 \\
\bottomrule
\end{tabular}



In [21]:
# ==================================================
# ADIM 18C: XGBoost Feature Importance Tablosu
# ==================================================

xgb_latex_df = xgb_importance_df.copy()
xgb_latex_df["Importance"] = xgb_latex_df["Importance"].map(lambda x: f"{x:.4f}")

xgb_latex_df = xgb_latex_df.rename(columns={
    "Feature": "Öznitelik",
    "Importance": "Önem Değeri"
})

print("XGBoost Feature Importance Tablosu:")
display(xgb_latex_df)

print("\nLaTeX kodu:\n")
print(xgb_latex_df.to_latex(index=False, escape=False))

XGBoost Feature Importance Tablosu:


,Öznitelik,Önem Değeri
0,zagreb_m1,0.3881
1,gutman,0.1577
2,zagreb_m2,0.1364
3,sombor,0.1327
4,randic,0.1011
5,wiener,0.0435
6,estrada,0.0405



LaTeX kodu:

\begin{tabular}{ll}
\toprule
Öznitelik & Önem Değeri \\
\midrule
zagreb_m1 & 0.3881 \\
gutman & 0.1577 \\
zagreb_m2 & 0.1364 \\
sombor & 0.1327 \\
randic & 0.1011 \\
wiener & 0.0435 \\
estrada & 0.0405 \\
\bottomrule
\end{tabular}



In [22]:
# ==================================================
# ADIM 18D: SVM Permutation Importance Tablosu
# ==================================================

svm_latex_df = svm_perm_importance_df.copy()

svm_latex_df["Importance_Mean"] = svm_latex_df["Importance_Mean"].map(lambda x: f"{x:.4f}")
svm_latex_df["Importance_Std"] = svm_latex_df["Importance_Std"].map(lambda x: f"{x:.4f}")

svm_latex_df = svm_latex_df.rename(columns={
    "Feature": "Öznitelik",
    "Importance_Mean": "Ortalama Önem",
    "Importance_Std": "Standart Sapma"
})

print("SVM Permutation Importance Tablosu:")
display(svm_latex_df)

print("\nLaTeX kodu:\n")
print(svm_latex_df.to_latex(index=False, escape=False))

SVM Permutation Importance Tablosu:


,Öznitelik,Ortalama Önem,Standart Sapma
0,zagreb_m2,0.0383,0.0441
1,sombor,0.0142,0.0308
2,zagreb_m1,0.0142,0.0349
3,gutman,0.0057,0.0219
4,estrada,0.0043,0.0236
5,wiener,-0.0113,0.0244
6,randic,-0.0135,0.0178



LaTeX kodu:

\begin{tabular}{lll}
\toprule
Öznitelik & Ortalama Önem & Standart Sapma \\
\midrule
zagreb_m2 & 0.0383 & 0.0441 \\
sombor & 0.0142 & 0.0308 \\
zagreb_m1 & 0.0142 & 0.0349 \\
gutman & 0.0057 & 0.0219 \\
estrada & 0.0043 & 0.0236 \\
wiener & -0.0113 & 0.0244 \\
randic & -0.0135 & 0.0178 \\
\bottomrule
\end{tabular}



In [23]:
# ==================================================
# ADIM 18E: Classification Report Tablosu
# ==================================================

from sklearn.metrics import classification_report

report_dict = classification_report(
    y,
    y_pred_cv,
    target_names=["Sınıf 0", "Sınıf 1"],
    output_dict=True
)

report_df = pd.DataFrame(report_dict).T

# Sadece gerekli satırları alalım
report_df = report_df.loc[
    ["Sınıf 0", "Sınıf 1", "accuracy", "macro avg", "weighted avg"]
]

# Accuracy satırında bazı sütunlar NaN olabileceği için düzenleyelim
report_df = report_df.reset_index().rename(columns={"index": "Sınıf / Ortalama"})

# Sayısal değerleri formatlayalım
for col in ["precision", "recall", "f1-score"]:
    report_df[col] = report_df[col].map(lambda x: f"{x:.2f}" if pd.notnull(x) else "")

report_df["support"] = report_df["support"].map(lambda x: f"{int(x)}" if pd.notnull(x) else "")

report_df = report_df.rename(columns={
    "precision": "Precision",
    "recall": "Recall",
    "f1-score": "F1-Score",
    "support": "Support"
})

print("Classification Report Tablosu:")
display(report_df)

print("\nLaTeX kodu:\n")
print(report_df.to_latex(index=False, escape=False))

Classification Report Tablosu:


,Sınıf / Ortalama,Precision,Recall,F1-Score,Support
0,Sınıf 0,0.81,0.86,0.83,63
1,Sınıf 1,0.93,0.90,0.91,125
2,accuracy,0.88,0.88,0.88,0
3,macro avg,0.87,0.88,0.87,188
4,weighted avg,0.89,0.88,0.88,188



LaTeX kodu:

\begin{tabular}{lllll}
\toprule
Sınıf / Ortalama & Precision & Recall & F1-Score & Support \\
\midrule
Sınıf 0 & 0.81 & 0.86 & 0.83 & 63 \\
Sınıf 1 & 0.93 & 0.90 & 0.91 & 125 \\
accuracy & 0.88 & 0.88 & 0.88 & 0 \\
macro avg & 0.87 & 0.88 & 0.87 & 188 \\
weighted avg & 0.89 & 0.88 & 0.88 & 188 \\
\bottomrule
\end{tabular}



In [24]:
# ==================================================
# ADIM 18F: Confusion Matrix Tablosu
# ==================================================

cm_latex_df = cm_df.copy()

print("Confusion Matrix Tablosu:")
display(cm_latex_df)

print("\nLaTeX kodu:\n")
print(cm_latex_df.to_latex(index=True, escape=False))

Confusion Matrix Tablosu:


,Tahmin 0,Tahmin 1
Gerçek 0,54,9
Gerçek 1,13,112



LaTeX kodu:

\begin{tabular}{lrr}
\toprule
 & Tahmin 0 & Tahmin 1 \\
\midrule
Gerçek 0 & 54 & 9 \\
Gerçek 1 & 13 & 112 \\
\bottomrule
\end{tabular}

